# T-Rex — validation-set inference analysis (latency, error, attention)

Run this against a **fine-tuned origami checkpoint** you already have saved to
Drive (produced by `trex_colab.ipynb`'s training sections). It does three
things over the first `NUM_EPISODES` held-out episodes (controllable below):

1. **Error + latency + safety** — reuses `scripts/eval_origami.py` as a
   subprocess (same as `trex_colab.ipynb`'s eval cells), so the numbers are
   computed by the exact, already-verified deployed-inference code path
   (`forward_flow_action_partial` + `tactile_flow_continue`), not a
   reimplementation.
2. **Attention maps** — a separate pass reusing `scripts/train.py`'s
   `run_validation(..., capture_attention=True)`, the same mechanism used
   during training's periodic validation, over the same episodes.
3. Inline plots for all of the above.

**Two small fixes landed in `eval_origami.py` alongside this notebook** (both
requested/verified in the surrounding session, not part of this file):

- `--latency_samples`'s timer used to start *after* `build_embeds()` — which
  runs `prepare_inputs_embeds`, i.e. the vision-tower (ViT) forward pass plus
  the host→device transfer — had already returned. That silently excluded
  vision-embedding cost from both `slow_tick` and `total`, understating the
  number the comment next to it literally calls "the number the robot waits
  on." Fixed: the timer now starts before `build_embeds()`, a new
  `vision_embed` phase is reported, and `total` is genuinely end-to-end.
  `total_compute_only` keeps the old (vision-excluded) definition around so a
  run measured before this fix is still comparable.
- `--first_k_episodes K`: restricts scoring (and the episode traces) to a
  fixed prefix of episodes instead of `--num_eval_samples`'s evenly-spaced
  sampling over the whole split — what this notebook uses for a fast,
  reproducible, controllable-size slice.

**Path layout** (mirrors `trex_colab.ipynb`):
- Code: `/content/origami_trex/T-Rex`
- Checkpoint (unzipped from Drive): `/content/checkpoint`
- Val data: wherever your prepared `origami_flat/{pilot,full}/val` already is
- Outputs: `/content/drive/MyDrive/iros2026/eval_analysis`

**Security note:** `trex_colab.ipynb`'s clone cell has a live GitHub PAT typed
directly into the notebook. This notebook does not copy that token — it reads
`GITHUB_PAT` from a Colab secret (or an env var you set yourself) instead, so
cloning still works but the credential isn't duplicated into a second file.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Setup — code + deps

Skips backbone/checkpoint downloads (unlike `colab_setup.sh`) — this notebook
loads everything from your already-fine-tuned checkpoint (see §2), which is
self-contained (`processor/`, `config.json`, `model.pt`).


In [ ]:
import os

PROJECT_ROOT = "/content/origami_trex/T-Rex"

if not os.path.isdir(PROJECT_ROOT):
    # Prefer a Colab secret (Runtime -> secrets icon in the left sidebar) over
    # typing a token into a cell -- avoids putting a second live credential
    # into a file that might get saved/shared. Falls back to an env var, then
    # an empty string (clone will fail with a clear 404 rather than hang).
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_PAT") or ""
    except Exception:
        pass
    token = token or os.environ.get("GITHUB_PAT", "")
    if not token:
        print("No GITHUB_PAT secret/env var found. Either add one (Colab "
              "secrets icon) or clone manually with the token from "
              "trex_colab.ipynb's own clone cell, then re-run this cell.")
    auth = f"{token}@" if token else ""
    !git clone https://{auth}github.com/KshitijBhat/origami_trex.git /content/origami_trex
else:
    print(f"already present: {PROJECT_ROOT}")


In [ ]:
%%bash
command -v uv >/dev/null 2>&1 || pip install -q uv
UV_PY="$(command -v python)"
uv pip install --python "${UV_PY}" -q \
    torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "${UV_PY}" -q -e /content/origami_trex/T-Rex
uv pip install --python "${UV_PY}" -q bitsandbytes


## 2. Checkpoint + run parameters

Three ways to point this at a checkpoint, in priority order:

1. **Already have an unzipped checkpoint dir on Drive** (e.g. Drive's own
   `.../trex_origami_fold_plane/<run>/checkpoint-0-5/` with `model.pt`,
   `processor/`, `config.json` sitting right in it) — set `CHECKPOINT_DIR` to
   that exact path and leave `CHECKPOINT_ZIP` empty.
2. **Don't know/want to type the exact nested Drive path** — leave
   `CHECKPOINT_DIR` empty and set `CHECKPOINT_NAME` to the folder's name (e.g.
   `"checkpoint-0-5"`); the next cell searches your whole Drive for a folder
   with that name containing `model.pt`.
3. **Only have a zip** — set `CHECKPOINT_ZIP` to its Drive path; it gets
   unzipped into `CHECKPOINT_DIR` (created if needed) before the same
   `model.pt` search runs.

`NUM_EPISODES` is the `K` in "first K validation episodes" — every section
below (error/latency/safety, traces, attention) is confined to exactly these
episodes, so the numbers you see are always about the same fixed slice.


In [ ]:
CHECKPOINT_ZIP = ""                     # only if you have a zip, not a dir (see §2 above)
CHECKPOINT_DIR = ""                     # exact path if you know it, e.g.
                                         # "/content/drive/MyDrive/iros2026/outputs/trex_origami_fold_plane/<run>/checkpoint-0-5"
CHECKPOINT_NAME = "checkpoint-0-5"      # used to auto-locate only when CHECKPOINT_DIR is left empty
BASE_MODEL_PATH = ""   # only needed if the checkpoint's own config.json is missing

VAL_ROOT = "/content/data/origami_flat/pilot/val"
URDF_PATH = "/content/urdf/north_poc2_2_v3_1.urdf"

NUM_EPISODES = 5            # K -- first K held-out episodes, controllable
EVAL_BATCH_SIZE = 8
LATENCY_SAMPLES = 20        # batch-1 timed requests within those K episodes
TRACE_EPISODES = min(NUM_EPISODES, 5)
NUM_WORKERS = 2             # 0 rules out Drive-FUSE / pyarrow-fork dataloader
                            # weirdness in Colab if a run seems to hang -- the
                            # tradeoff is slower data loading, not correctness

OUT_DIR = f"/content/drive/MyDrive/iros2026/eval_analysis/first_{NUM_EPISODES}ep"
ATTN_VIZ_DIR = os.path.join(OUT_DIR, "attention")
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
import glob

if CHECKPOINT_ZIP:
    CHECKPOINT_DIR = CHECKPOINT_DIR or "/content/checkpoint"
    if not glob.glob(os.path.join(CHECKPOINT_DIR, "**", "model.pt"), recursive=True):
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)
        print(f"unzipping {CHECKPOINT_ZIP} -> {CHECKPOINT_DIR}")
        !unzip -q -o "{CHECKPOINT_ZIP}" -d "{CHECKPOINT_DIR}"
elif not CHECKPOINT_DIR:
    # No exact path given -- search Drive for a folder named CHECKPOINT_NAME
    # rather than requiring you to hand-copy Drive's (often truncated-in-the-UI)
    # nested path. Slow-ish the first time (walks Drive over FUSE); cheap after,
    # since CHECKPOINT_DIR is now resolved for the rest of this session.
    print(f"searching Drive for a '{CHECKPOINT_NAME}' folder (this can take a bit)...")
    candidates = glob.glob(f"/content/drive/MyDrive/**/{CHECKPOINT_NAME}", recursive=True)
    candidates = [c for c in candidates if os.path.isdir(c)
                 and glob.glob(os.path.join(c, "**", "model.pt"), recursive=True)]
    assert candidates, (f"no folder named '{CHECKPOINT_NAME}' with a model.pt inside it "
                        f"found under /content/drive/MyDrive -- double check the name, or "
                        f"just set CHECKPOINT_DIR directly instead.")
    if len(candidates) > 1:
        print(f"WARNING: {len(candidates)} matches, using the first -- set "
              f"CHECKPOINT_DIR explicitly if this isn't the one you want:")
        for c in candidates:
            print(" ", c)
    CHECKPOINT_DIR = candidates[0]

# The zip might have the checkpoint's contents at its root, or nested one
# level under the checkpoint's own dirname -- same flattening eval_origami.py
# users hit with the backbone bundle. Find model.pt and repoint CHECKPOINT_DIR
# at whichever directory actually holds it.
hits = glob.glob(os.path.join(CHECKPOINT_DIR, "**", "model.pt"), recursive=True)
assert hits, f"no model.pt under {CHECKPOINT_DIR} -- check CHECKPOINT_ZIP/CHECKPOINT_DIR"
CHECKPOINT_DIR = os.path.dirname(hits[0])
print("CHECKPOINT_DIR ->", CHECKPOINT_DIR)
print("  processor/:", os.path.isdir(os.path.join(CHECKPOINT_DIR, "processor")))
print("  config.json:", os.path.exists(os.path.join(CHECKPOINT_DIR, "config.json")))
print("  training_args.json:", os.path.exists(os.path.join(CHECKPOINT_DIR, "training_args.json")))


## 3. Error + latency + safety

Runs `eval_origami.py` as a subprocess (its own process, so the ~4-5 GB of
GPU memory it uses is released the moment this cell finishes — the attention
pass in §6 loads its own model instance afterward, not concurrently).
`PYTHONUNBUFFERED=1` matters here: stdout is a pipe under `%%bash`, not a
tty, so without it CPython block-buffers every `print()` and the progress
lines (`"N/M samples | Ts"` etc.) only show up in one dump at the very end —
this cell can look frozen for the entire run otherwise, even though it's
working the whole time.

Reads out: MAE/RMSE per mode (`cascaded` vs `blind`, plus the `hold_state` /
`repeat_command` naive baselines and the `teleop_gt` safety-floor row), the
horizon curve, a contact-only split, safety/URDF violation rates, and now a
4-phase latency breakdown (`vision_embed` / `slow_tick` / `fast_tick` /
`total`).

**For a fast first pass**, `--trace_episodes` is the biggest add-on cost —
it reruns `trace_samples` (300 by default) *more* rows per episode through
the full cascaded pipeline purely for the episode-trace plot, on top of the
main scoring loop. Set `TRACE_EPISODES = 0` and/or lower `LATENCY_SAMPLES`
in §2 if `NUM_EPISODES` alone isn't making this quick enough.

Runs via `subprocess` + a manual line-read loop rather than `%%bash` —
`%%bash`'s own output capture doesn't reliably stream live in every
Jupyter/Colab combination (some only flush once the whole subprocess exits,
`PYTHONUNBUFFERED` or not). Printing each line through the notebook kernel's
own stdout sidesteps that entirely; Colab always renders that live.


In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "scripts/eval_origami.py",
    "--checkpoint_path", CHECKPOINT_DIR,
    "--origami_root", VAL_ROOT,
    "--out_dir", OUT_DIR,
    "--urdf", URDF_PATH,
    "--first_k_episodes", str(NUM_EPISODES),
    "--batch_size", str(EVAL_BATCH_SIZE),
    "--modes", "cascaded", "blind",
    "--latency_samples", str(LATENCY_SAMPLES),
    "--trace_episodes", str(TRACE_EPISODES),
    "--num_workers", str(NUM_WORKERS),
]
env = {**os.environ, "PYTHONPATH": "/content/origami_trex/T-Rex", "PYTHONUNBUFFERED": "1"}

proc = subprocess.Popen(cmd, cwd="/content/origami_trex/T-Rex", env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"eval_origami.py exited with code {proc.returncode}")


In [ ]:
import json

with open(os.path.join(OUT_DIR, "metrics.json")) as f:
    metrics = json.load(f)

print(f"{'mode':<16} {'MAE(deg)':>10} {'RMSE(deg)':>10} {'MSE':>10} {'MAE contact':>12}")
print("-" * 62)
for name, report in metrics["results"].items():
    overall, contact = report.get("overall", {}), report.get("contact", {})
    print(f"{name:<16} {overall.get('mae_deg', float('nan')):>10.3f} "
          f"{overall.get('rmse_deg', float('nan')):>10.3f} "
          f"{overall.get('mse', float('nan')):>10.5f} "
          f"{contact.get('mae_deg', float('nan')):>12.3f}")

print()
print(f"{'phase':<20} {'mean(ms)':>10} {'p50(ms)':>10} {'p95(ms)':>10}")
print("-" * 52)
for phase, v in metrics.get("request_latency_ms", {}).items():
    print(f"{phase:<20} {v['mean']:>10.1f} {v['p50']:>10.1f} {v['p95']:>10.1f}")


In [ ]:
import matplotlib.pyplot as plt

lat = metrics.get("request_latency_ms", {})
phases = [p for p in ("vision_embed", "slow_tick", "fast_tick") if p in lat]
means = [lat[p]["mean"] for p in phases]
p95s = [lat[p]["p95"] for p in phases]

fig, ax = plt.subplots(figsize=(6, 4))
x = range(len(phases))
ax.bar(x, means, color=["#4C72B0", "#DD8452", "#55A868"][:len(phases)], label="mean")
ax.plot(x, p95s, "ko--", label="p95")
ax.set_xticks(list(x)); ax.set_xticklabels(phases)
ax.set_ylabel("ms")
ax.set_title(f"request latency at batch 1 (n={LATENCY_SAMPLES})")
ax.axhline(540, color="gray", linestyle=":", label="kit's pi0.5 reference (~540 ms)")
ax.legend()
fig.tight_layout()
plt.show()

total = lat.get("total", {}).get("mean")
compute_only = lat.get("total_compute_only", {}).get("mean")
if total and compute_only:
    print(f"vision-embedding adds {total - compute_only:.0f} ms "
          f"({100 * (total - compute_only) / total:.0f}% of the real total) "
          f"on top of the compute-only number this used to report.")


In [ ]:
from IPython.display import Image, display

png_order = ["parity_plot.png", "horizon_error.png", "group_error.png",
             "group_error_distribution.png", "episode_trace.png"]
png_order += sorted(f for f in os.listdir(OUT_DIR)
                    if f.endswith(".png") and f not in png_order)
for name in png_order:
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(name)
        display(Image(filename=path))


## 4. Init the model directly (for §6's attention pass)

Same loader `eval_origami.py`/`test.py`'s serving path uses
(`test.model_load`), loaded by path for the same reason `eval_origami.py`
does it: `sys.path` order makes a plain `import test` ambiguous with
CPython's own top-level `test` package.


In [ ]:
import sys, importlib.util
from types import SimpleNamespace

sys.path.insert(0, "/content/origami_trex/T-Rex/scripts")
sys.path.insert(0, "/content/origami_trex/T-Rex")

import torch

def _load_by_path(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

test_mod = _load_by_path("trex_test_server", "/content/origami_trex/T-Rex/scripts/test.py")
eo = _load_by_path("trex_eval_origami", "/content/origami_trex/T-Rex/scripts/eval_origami.py")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

load_args = SimpleNamespace(
    checkpoint_path=CHECKPOINT_DIR, base_model_path=BASE_MODEL_PATH,
    stats_path="", dataset_name="",
    action_dim=65, action_chunk=25,
    use_robot_state=1, use_tactile_vec=1, use_tactile_deform=1,
    tactile_intermediate_size=0, n_flare_tokens_per_frame=0, n_flare_steps=0,
    use_tactile_code=0, vqvae_codebook_size=64, use_tactile_vqvae=0, vqvae_config=None,
    cascaded_total_steps=10, cascaded_split_step=6,
)
model, processor, statistic = test_mod.model_load(load_args)
model = model.to(device).eval()
print(f"model loaded: action_dim={load_args.action_dim} action_chunk={load_args.action_chunk} "
      f"n_flare_tokens={model.n_flare_tokens}")


## 5. Load the same first-K episodes

Same `dataset.index` prefix trick `--first_k_episodes` uses inside
`eval_origami.py` (`_build_index` appends every row of episode *e* before
episode *e+1*, so the first K episodes are a contiguous prefix, not a scan).


In [ ]:
from torch.utils.data import DataLoader, Subset

config = eo.dataset_config_from_checkpoint(CHECKPOINT_DIR, {})
dataset = eo.OrigamiDataset(config, processor, eo._Printer(), root=VAL_ROOT)

n_rows = sum(dataset.ep_rows[:NUM_EPISODES])
episode_subset = Subset(dataset, list(range(n_rows)))
print(f"first {NUM_EPISODES} episodes -> {n_rows} samples: "
      f"{[dataset.episodes[i]['file'] for i in range(NUM_EPISODES)]}")

# One batch is all §6 uses (attention capture only ever looks at batch i==0) --
# kept small on purpose so this cell finishes in seconds, not minutes.
attn_batch_size = min(4, n_rows)
attn_loader = DataLoader(episode_subset, batch_size=attn_batch_size, shuffle=False,
                         num_workers=0, collate_fn=dataset.collate_fn)


## 6. Attention maps

Reuses `scripts/train.py`'s `run_validation(..., capture_attention=True)` —
the exact mechanism validated during training, not a reimplementation.

**Scope, straight from `attention_capture.py`'s own docstring:** this call
runs the model's *training-style* forward pass (the one that computes the
action-expert full-flow loss over `model.model(...)`), not the *deployed*
inference path (`forward_flow_action_partial` + `tactile_flow_continue`)
that §3 just measured. It captures joint self/cross-attention among the
latent (vision/language), action, and tactile token groups for that forward
pass. It does **not** reach inside the cascaded tactile-refinement call
itself — see the docstring in `qwen_vla/attention_capture.py` for why
(extending there touches the real training-time cascaded loss, a
higher-stakes change than this validation-only utility).

`load_args.max_val_batches = 1` caps this to exactly the one batch
`capture_attention` actually looks at (`i == 0`) — nothing here needs a
second batch.


In [ ]:
from accelerate import Accelerator

train_mod = _load_by_path("trex_train", "/content/origami_trex/T-Rex/scripts/train.py")
accelerator = Accelerator()

load_args.max_val_batches = 1
os.makedirs(ATTN_VIZ_DIR, exist_ok=True)

K = model.n_flare_tokens          # total flare-query tokens, already baked into the model
use_flare = K > 0

# run_validation assumes accelerate already put batches on the right device --
# true in train.py's own main() (line ~1055: accelerator.prepare(model, optimizer,
# dataloader)), not true here since §4 moved the model with a plain .to(device)
# and attn_loader is a bare DataLoader. Without this, index_select-style ops
# inside the model see a CPU batch against a CUDA model and crash.
model, attn_loader = accelerator.prepare(model, attn_loader)

val_metrics = train_mod.run_validation(
    model, attn_loader, accelerator, load_args,
    is_stage1=False, use_flare=use_flare, K=K, T_per_frame=0, flare_layer_idx=0,
    capture_attention=True, viz_dir=ATTN_VIZ_DIR, capture_n_samples=attn_batch_size,
)
print(val_metrics)
print(f"saved to {ATTN_VIZ_DIR}:", sorted(os.listdir(ATTN_VIZ_DIR)))


In [ ]:
import matplotlib.image as mpimg

attn_pngs = sorted(f for f in os.listdir(ATTN_VIZ_DIR) if f.endswith(".png"))
n = len(attn_pngs)
if n == 0:
    print("no attention maps saved -- see the [viz] warning printed above, "
          "or attention_capture.py's scope note (nothing to capture when "
          "both latent/action index sets aren't both non-empty).")
else:
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = axes.flatten() if n > 1 else [axes]
    for ax, name in zip(axes, attn_pngs):
        ax.imshow(mpimg.imread(os.path.join(ATTN_VIZ_DIR, name)))
        ax.set_title(name.replace(".png", ""), fontsize=7)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


## What to read out of this

- **§3's `blind` vs `cascaded`** — same question `trex_colab.ipynb` §4 asks:
  does the tactile expert actually help on *these* episodes? If `blind`
  still wins (it did on the pilot checkpoint), that is the one number that
  should decide whether the tactile expert ships at all.
- **§3's latency bar** — `vision_embed` was silently missing from `total`
  before this notebook's `eval_origami.py` fix. If it's a large fraction of
  the bar, that's a second, independent lever beyond flow-matching step
  count (fewer/cheaper vision tokens, e.g. a smaller `--image_size` or
  caching the slow tick's vision embedding across the fast ticks that follow
  it in the same chunk, which today's code does not do).
- **§6's attention maps** — `latent_to_latent` answers whether the model
  looks at the fold region at all; `action_to_latent` answers whether the
  action expert actually conditions on vision or mostly attends to its own
  tokens (the same question this project asked of the vitacformer++ ACT
  policy). Compare a middle layer against the last layer — attention that
  only sharpens in the final layer suggests late binding, which is a very
  different story from sharp attention throughout.
- Re-run with a larger `NUM_EPISODES` once a specific episode here looks
  interesting, to see whether the pattern holds or was specific to these K.
